# Rilla Data Analysis - Working Notebook

**Working copy for project development**

This notebook loads local CSV files for analysis, with size checks and optimized loading for large files.

## Files to Load:
- `comments_data.csv` (~141MB)
- `sales_rep_recordings_data.csv` (~1.5GB) ⚠️ Large file
- `sales_rep_identity.csv` (~7.8MB)
- `manager_data.csv` (~2.4MB)


In [1]:
# Install necessary packages (run once)
import sys
import subprocess

def install_package(package):
    """Install a package if not already installed"""
    try:
        __import__(package)
        print(f"✓ {package} is already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✓ {package} installed successfully")

# Install required packages
packages = ['pandas', 'numpy']
for package in packages:
    install_package(package)

print("\n✅ All packages ready!")


✓ pandas is already installed
✓ numpy is already installed

✅ All packages ready!


In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path

print("✅ Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


✅ Libraries imported successfully
Pandas version: 2.3.3
NumPy version: 2.0.2


In [3]:
# Define file paths
base_path = Path('/Users/winstonfeng/Downloads/Rilla/Rilla data')

files = {
    'comments_data': base_path / 'comments_data.csv',
    'sales_rep_recordings_data': base_path / 'sales_rep_recordings_data.csv',
    'sales_rep_identity': base_path / 'sales_rep_identity.csv',
    'manager_data': base_path / 'manager_data.csv'
}

# Check file sizes
print("📊 File Size Information:\n")
for name, filepath in files.items():
    if filepath.exists():
        size_bytes = os.path.getsize(filepath)
        size_mb = size_bytes / (1024 * 1024)
        size_gb = size_bytes / (1024 * 1024 * 1024)
        if size_gb >= 1:
            print(f"{name}: {size_gb:.2f} GB ({size_mb:.2f} MB)")
        else:
            print(f"{name}: {size_mb:.2f} MB")
    else:
        print(f"❌ {name}: File not found at {filepath}")


📊 File Size Information:

comments_data: 140.73 MB
sales_rep_recordings_data: 1.55 GB (1582.72 MB)
sales_rep_identity: 7.80 MB
manager_data: 2.35 MB


In [4]:
# Load smaller files (under 200MB) directly into memory
print("📥 Loading smaller files...\n")

# Load manager_data.csv (2.4MB)
print("Loading manager_data.csv...")
manager_data = pd.read_csv(files['manager_data'])
print(f"✓ manager_data loaded: {manager_data.shape[0]:,} rows × {manager_data.shape[1]} columns")
print(f"  Memory usage: {manager_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB\n")

# Load sales_rep_identity.csv (7.8MB)
print("Loading sales_rep_identity.csv...")
sales_rep_identity = pd.read_csv(files['sales_rep_identity'])
print(f"✓ sales_rep_identity loaded: {sales_rep_identity.shape[0]:,} rows × {sales_rep_identity.shape[1]} columns")
print(f"  Memory usage: {sales_rep_identity.memory_usage(deep=True).sum() / 1024**2:.2f} MB\n")

# Load comments_data.csv (141MB) - still manageable but monitor memory
print("Loading comments_data.csv (141MB)...")
comments_data = pd.read_csv(files['comments_data'])
print(f"✓ comments_data loaded: {comments_data.shape[0]:,} rows × {comments_data.shape[1]} columns")
print(f"  Memory usage: {comments_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB\n")

print("✅ Smaller files loaded successfully!")


📥 Loading smaller files...

Loading manager_data.csv...
✓ manager_data loaded: 17,553 rows × 8 columns
  Memory usage: 7.67 MB

Loading sales_rep_identity.csv...
✓ sales_rep_identity loaded: 61,520 rows × 6 columns
  Memory usage: 23.94 MB

Loading comments_data.csv (141MB)...
✓ comments_data loaded: 632,461 rows × 8 columns
  Memory usage: 378.96 MB

✅ Smaller files loaded successfully!


In [5]:
# Load large file (sales_rep_recordings_data.csv ~1.5GB)
print("📥 Loading large file: sales_rep_recordings_data.csv (~1.5GB)")
print("This may take a few minutes...\n")

# Method 1: Direct loading (recommended for M4 with 16GB RAM)
# This should work fine with your system specs
try:
    print("Loading directly into memory...")
    sales_rep_recordings_data = pd.read_csv(files['sales_rep_recordings_data'], low_memory=False)
    print(f"✓ sales_rep_recordings_data loaded: {sales_rep_recordings_data.shape[0]:,} rows × {sales_rep_recordings_data.shape[1]} columns")
    print(f"  Memory usage: {sales_rep_recordings_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"  Columns: {list(sales_rep_recordings_data.columns)}")
    
except MemoryError:
    print("⚠️ Direct loading failed! Trying chunking approach...")
    # Method 2: Chunked loading (if direct loading fails)
    chunk_size = 50000  # Process in chunks of 50k rows
    chunks = []
    row_count = 0
    
    print("Loading in chunks...")
    for i, chunk in enumerate(pd.read_csv(files['sales_rep_recordings_data'], chunksize=chunk_size, low_memory=False)):
        chunks.append(chunk)
        row_count += len(chunk)
        if i == 0:
            print(f"  Columns: {list(chunk.columns)}")
            print(f"  First chunk shape: {chunk.shape}")
        if (i + 1) % 10 == 0:
            print(f"  Processed {row_count:,} rows... ({len(chunks)} chunks)")
    
    # Combine chunks for full dataset
    print("\nCombining chunks...")
    sales_rep_recordings_data = pd.concat(chunks, ignore_index=True)
    print(f"✓ sales_rep_recordings_data loaded: {sales_rep_recordings_data.shape[0]:,} rows × {sales_rep_recordings_data.shape[1]} columns")
    print(f"  Memory usage: {sales_rep_recordings_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
print("\n✅ Large file loaded successfully!")


📥 Loading large file: sales_rep_recordings_data.csv (~1.5GB)
This may take a few minutes...

Loading directly into memory...
✓ sales_rep_recordings_data loaded: 8,541,723 rows × 7 columns
  Memory usage: 4343.79 MB
  Columns: ['recording_id', 'user_id', 'account_id', 'hubspot_company_id', 'recording_ts', 'recording_length', 'uploaded_at']

✅ Large file loaded successfully!


In [6]:
# Alternative: Load large file directly (if you have enough memory)
# Uncomment below if the chunking approach above doesn't work or if you want full control

# print("Loading sales_rep_recordings_data.csv directly...")
# sales_rep_recordings_data = pd.read_csv(files['sales_rep_recordings_data'])
# print(f"✓ Loaded: {sales_rep_recordings_data.shape[0]:,} rows × {sales_rep_recordings_data.shape[1]} columns")
# print(f"Memory usage: {sales_rep_recordings_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


In [7]:
# Display summary of all loaded datasets
print("=" * 60)
print("📊 DATA SUMMARY")
print("=" * 60)

datasets = {
    'manager_data': manager_data,
    'sales_rep_identity': sales_rep_identity,
    'comments_data': comments_data,
    'sales_rep_recordings_data': sales_rep_recordings_data
}

for name, df in datasets.items():
    print(f"\n{name.upper().replace('_', ' ')}:")
    print(f"  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"  Columns: {list(df.columns)}")

print("\n" + "=" * 60)


📊 DATA SUMMARY

MANAGER DATA:
  Shape: 17,553 rows × 8 columns
  Memory: 7.67 MB
  Columns: ['user_id', 'account_id', 'hubspot_company_id', 'role', 'created_at', 'is_removed', 'onboarding_completed_at', 'ridealong_comments_given']

SALES REP IDENTITY:
  Shape: 61,520 rows × 6 columns
  Memory: 23.94 MB
  Columns: ['user_id', 'account_id', 'hubspot_company_id', 'role', 'is_removed', 'created_at']

COMMENTS DATA:
  Shape: 632,461 rows × 8 columns
  Memory: 378.96 MB
  Columns: ['ridealong_comment_id', 'ridealong_id', 'manager_id', 'account_id', 'hubspot_company_id', 'conversation_id', 'comment_ts', 'comment_text_length']

SALES REP RECORDINGS DATA:
  Shape: 8,541,723 rows × 7 columns
  Memory: 4343.79 MB
  Columns: ['recording_id', 'user_id', 'account_id', 'hubspot_company_id', 'recording_ts', 'recording_length', 'uploaded_at']



In [8]:
# Display first few rows of each dataset
print("📋 PREVIEW OF DATASETS\n")

for name, df in datasets.items():
    print(f"{'='*60}")
    print(f"{name.upper().replace('_', ' ')} - First 3 rows:")
    print(f"{'='*60}")
    print(df.head(3))
    print("\n")


📋 PREVIEW OF DATASETS

MANAGER DATA - First 3 rows:
                                user_id                            account_id  \
0  0002afd4-c64e-496b-9deb-2c4d43ed5f0f  844972e0-7e71-4392-bcf2-99e40eaae761   
1  00054e19-4320-48f9-9159-e3fa6ff0fc97  2b8092cb-4219-4eff-bb2e-7e6b5f00ab4b   
2  001507c7-b454-4713-a094-115f4e99077c  cc9185e3-3506-46f6-8e08-636534992311   

  hubspot_company_id         role                         created_at  \
0        20849882115        admin  2024-05-22 14:57:29.041496 +00:00   
1        21979438907  teamManager  2025-07-07 16:34:41.872344 +00:00   
2        36544810303  teamManager  2025-10-24 02:49:27.042525 +00:00   

   is_removed onboarding_completed_at  ridealong_comments_given  
0       False              2024-05-31                         0  
1       False              2025-07-16                       109  
2       False                     NaN                         2  


SALES REP IDENTITY - First 3 rows:
                                u

In [9]:
# Check data types and missing values
print("🔍 DATA QUALITY CHECK\n")

for name, df in datasets.items():
    print(f"{'='*60}")
    print(f"{name.upper().replace('_', ' ')}:")
    print(f"{'='*60}")
    print("\nData Types:")
    print(df.dtypes)
    print("\nMissing Values:")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("No missing values!")
    print("\n" + "-"*60 + "\n")


🔍 DATA QUALITY CHECK

MANAGER DATA:

Data Types:
user_id                     object
account_id                  object
hubspot_company_id          object
role                        object
created_at                  object
is_removed                    bool
onboarding_completed_at     object
ridealong_comments_given     int64
dtype: object

Missing Values:
hubspot_company_id           568
onboarding_completed_at    13045
dtype: int64

------------------------------------------------------------

SALES REP IDENTITY:

Data Types:
user_id               object
account_id            object
hubspot_company_id    object
role                  object
is_removed              bool
created_at            object
dtype: object

Missing Values:
hubspot_company_id    930
dtype: int64

------------------------------------------------------------

COMMENTS DATA:

Data Types:
ridealong_comment_id     object
ridealong_id             object
manager_id               object
account_id               object
hu

## Notes:

1. **Memory Management**: The large file (~1.5GB) is loaded directly. With 16GB RAM, this should work, but monitor memory usage.

2. **If Memory Issues Occur**: 
   - Use chunking: `pd.read_csv(file, chunksize=10000)` and process in chunks
   - Load only specific columns: `pd.read_csv(file, usecols=['col1', 'col2'])`
   - Consider using Dask for out-of-core processing

3. **Optimization Tips**:
   - Use appropriate data types (convert objects to categories if possible)
   - Parse dates during loading: `pd.read_csv(file, parse_dates=['date_column'])`
   - Consider using `low_memory=False` for consistent dtypes

4. **All dataframes are now available**:
   - `manager_data`
   - `sales_rep_identity`
   - `comments_data`
   - `sales_rep_recordings_data`
